<a href="https://colab.research.google.com/github/Saliyah-53/saliyah-stanceeval2026/blob/main/SaliAI_3Ideas_ALLaM_Track1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SaliAI — Three Advanced Ideas — ALLaM (Track 1)

We test three systems on ALLaM (our best model on Track 1):
1. **Sarcasm-aware:** detect sarcasm first, then classify.
2. **Mixture-of-experts (MoE):** three specialist prompts (dialect / sarcasm /
   political) + voting.
3. **Judged debate:** pro advocate + con advocate + judge.

Input: `test_seen.csv`. Requires a T4 GPU. Run all.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.5 MB/s eta 0:00:00


In [ ]:
import torch, os, re, zipfile
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd
assert torch.cuda.is_available(), "فعّل GPU"
print("GPU:", torch.cuda.get_device_name(0))
DATA_DIR="./data"; OUT="./ideas_allam_track1_out"; os.makedirs(OUT, exist_ok=True)
TEXT_COL="text"; TARGET_COL="target"

df=pd.read_csv(f"{DATA_DIR}/test_seen.csv", keep_default_na=False)
if "tweet_text" in df.columns and TEXT_COL not in df.columns: df=df.rename(columns={"tweet_text":TEXT_COL})
print("عدد التغريدات:", len(df))

GPU: Tesla T4
عدد التغريدات: 352


In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok=AutoTokenizer.from_pretrained("humain-ai/ALLaM-7B-Instruct-preview", trust_remote_code=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
mdl=AutoModelForCausalLM.from_pretrained("humain-ai/ALLaM-7B-Instruct-preview", quantization_config=bnb, device_map="auto", trust_remote_code=True)
mdl.eval()
def gen(messages, max_new=6, sample=False, temp=0.7):
    p=tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    e=tok(p, return_tensors="pt", return_token_type_ids=False).to(mdl.device)
    with torch.no_grad():
        o=mdl.generate(**e, max_new_tokens=max_new, do_sample=sample,
                       temperature=temp if sample else None, top_p=0.95 if sample else None,
                       pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][e["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
def parse(o):
    o=o.strip().lower()
    poss=[(o.rfind("against"),"Against"),(o.rfind("favor"),"Favor"),(o.rfind("none"),"None"),
          (o.rfind("معارض"),"Against"),(o.rfind("مؤيد"),"Favor"),(o.rfind("محايد"),"None")]
    poss=[(i,l) for i,l in poss if i>=0]
    return max(poss)[1] if poss else "None"
def save_sub(preds,name):
    txt=f"{OUT}/{name}.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
    with zipfile.ZipFile(f"{OUT}/{name}.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission.txt")
    print(f"{name}: {Counter(preds)}")

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
#idea 1
def sarcasm_aware(text, target):
    tline="الهدف: قيادة المرأة للسيارة"

    m1=[{"role":"system","content":"أجب بنعم أو لا فقط: هل التغريدة تحتوي سخرية أو تهكّم؟"},
        {"role":"user","content":f"التغريدة: {text}"}]
    sarc="نعم" in gen(m1, max_new=4)

    hint=("تنبيه: التغريدة ساخرة/تهكّمية، فقد يكون الموقف الحقيقي عكس ظاهر الكلمات. " if sarc else "")
    sysp=("أنت مصنّف مواقف عربي دقيق. "+hint+
          "أجب بكلمة واحدة: Favor أو Against أو None. Favor=مؤيد، Against=معارض، None=لا موقف واضح.")
    m2=[{"role":"system","content":sysp},
        {"role":"user","content":f"{tline}\nالتغريدة: {text}\nالموقف:"}]
    return parse(gen(m2, max_new=6))
preds1=[sarcasm_aware(str(r[TEXT_COL]),str(r[TARGET_COL])) for _,r in tqdm(df.iterrows(),total=len(df),desc="sarcasm-aware")]
save_sub(preds1,"sub_allam_idea1_sarcasm")

sarcasm-aware:   0%|          | 0/352 [00:00<?, ?it/s]

sub_allam_idea1_sarcasm: Counter({'Favor': 262, 'Against': 67, 'None': 23})


In [ ]:
#idea2
EXPERTS=[
 "أنت خبير في اللهجات العربية العامية. حلّل التعبيرات الدارجة وحدّد الموقف. أجب بكلمة: Favor أو Against أو None.",
 "أنت خبير في كشف السخرية والتهكّم. انتبه إن الموقف قد يكون عكس الظاهر. أجب بكلمة: Favor أو Against أو None.",
 "أنت محلل خطاب اجتماعي/سياسي. ميّز رأي الكاتب عن اقتباسه. أجب بكلمة: Favor أو Against أو None.",
]
def moe(text,target):
    tline="الهدف: قيادة المرأة للسيارة"
    votes=[]
    for sysp in EXPERTS:
        m=[{"role":"system","content":sysp},{"role":"user","content":f"{tline}\nالتغريدة: {text}\nالموقف:"}]
        votes.append(parse(gen(m, max_new=6)))
    return Counter(votes).most_common(1)[0][0]
preds2=[moe(str(r[TEXT_COL]),str(r[TARGET_COL])) for _,r in tqdm(df.iterrows(),total=len(df),desc="MoE-experts")]
save_sub(preds2,"sub_allam_idea2_experts")

MoE-experts:   0%|          | 0/352 [00:00<?, ?it/s]

sub_allam_idea2_experts: Counter({'Favor': 243, 'Against': 95, 'None': 14})


In [ ]:
#idea3
def debate(text,target):
    tline="الهدف: قيادة المرأة للسيارة"
    pro=gen([{"role":"system","content":"أنت محامٍ. اكتب جملة واحدة تدافع بها عن أن الكاتب مؤيد للهدف."},
             {"role":"user","content":f"{tline}\nالتغريدة: {text}"}], max_new=50)
    con=gen([{"role":"system","content":"أنت محامٍ. اكتب جملة واحدة تدافع بها عن أن الكاتب معارض للهدف."},
             {"role":"user","content":f"{tline}\nالتغريدة: {text}"}], max_new=50)
    judge=[{"role":"system","content":"أنت حَكَم محايد. بناءً على الحجتين والتغريدة، ما موقف الكاتب الحقيقي؟ أجب بكلمة: Favor أو Against أو None."},
           {"role":"user","content":f"{tline}\nالتغريدة: {text}\nحجة التأييد: {pro}\nحجة المعارضة: {con}\nالحكم:"}]
    return parse(gen(judge, max_new=6))
preds6=[debate(str(r[TEXT_COL]),str(r[TARGET_COL])) for _,r in tqdm(df.iterrows(),total=len(df),desc="debate")]
save_sub(preds6,"sub_allam_idea6_debate")
print("\n>>> ارفع الثلاثة على Track1 وقارن <<<")

debate:   0%|          | 0/352 [00:00<?, ?it/s]

sub_allam_idea6_debate: Counter({'Favor': 200, 'None': 104, 'Against': 48})

>>> ارفع الثلاثة على Track1 وقارن <<<
